In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel(r"original_dataset.xlsx")

df.head()

,Unnamed: 0,review_description,rating,company
0,0,سيئ جدا بعد الإصدار الجديد,-1,alahli_bank
1,1,ابلكيشن زباله بجد,-1,alahli_bank
2,2,سيئ التطبيق لايعمل,-1,alahli_bank
3,3,للأسف التطبيق للأسوأ كان جدا رائع وسهل وبسيط ا...,-1,alahli_bank
4,4,التحديث بطيئ جدا جدا عند الفتح,-1,alahli_bank


**Data Understanding**

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67127 entries, 0 to 67126
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Unnamed: 0          67127 non-null  int64 
 1   review_description  67125 non-null  object
 2   rating              67127 non-null  int64 
 3   company             67127 non-null  object
dtypes: int64(2), object(2)
memory usage: 2.0+ MB


In [4]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,67127.0,33563.000000,19378.040097,0.0,16781.5,33563.0,50344.5,67126.0
rating,67127.0,-0.040163,0.802836,-1.0,-1.0,0.0,1.0,1.0


In [16]:
df.isnull().sum()

Unnamed: 0            0
review_description    2
rating                0
company               0
dtype: int64

In [6]:
df.duplicated().sum()

0

In [29]:
df[df['review_description'].isna()]

,Unnamed: 0,review_description,rating,company
63890,63890,NaN,0,hotels
64405,64405,NaN,0,hotels


In [30]:
# dropping nulls

df.dropna(axis= 0, inplace= True)

df.isna().sum()

Unnamed: 0            0
review_description    0
rating                0
company               0
dtype: int64

In [31]:
# Unnamed: 0 for index in the excel file so we don't need it because we already have the index in our dataFrame

df.drop(columns= 'Unnamed: 0', inplace= True)

df.columns

Index(['review_description', 'rating', 'company'], dtype='object')

**Preprocessing**

My workFlow in preprocessing will be:

1- Normalization
2- Tokenize Arabic words
3- Remove stopwords
4- Stemming\Lemmatization
5- Rejoin tokens into clean string

**Normalization**

In [89]:
import re

def normalize(text):
    # Harkat
    arabic_diacritics_harakah = re.compile(r'[\u064B-\u0652]')
    text = re.sub(arabic_diacritics_harakah, '', text)

    # Repetitive char
    text = re.sub(r'(.)\1{3,}', r'\1', text)

    # replacing char 'أ,إ,آ' with 'ا'
    text = text.replace('آ', 'ا')
    text = text.replace('أ', 'ا')
    text = text.replace('إ', 'ا')
    # or I can use this to rplace them: --> text = re.sub(r'[آأإ]', 'ا', text)

    # Another replacing chars
    text = re.sub(r'[ى]', 'ي', text)  # Replace ى with ي
    text = re.sub(r'[ة]', 'ه', text)  # Replace ة with ه
    text = re.sub(r'[ؤ]', 'و', text)  # Replace ؤ with و
    text = re.sub(r'[ئ]', 'ي', text)  # Replace ئ with ي

    # Removing Tatweel 
    text = re.sub(r'[\u0640]', '', text)

    return text


# text = 'إلى حياةٍ رائعة فيها مؤيد و مليئة بالكككككفاءة و شئون و رئيس هيئة'
# normalize(text)

In [94]:
df['normalized'] = df['review_description'].apply(normalize)
print(df[['review_description', 'normalized']])

                                      review_description  \
0                             سيئ جدا بعد الإصدار الجديد   
1                                      ابلكيشن زباله بجد   
2                                     سيئ التطبيق لايعمل   
3      للأسف التطبيق للأسوأ كان جدا رائع وسهل وبسيط ا...   
4                         التحديث بطيئ جدا جدا عند الفتح   
...                                                  ...   
67122  كتاب جيد وإن كان مملا بعض الشيء عند منتصف الكت...   
67123  أول تجربة مع الخيال العلمي...الكثير من المعلوم...   
67124  مرضي. الافطار لذيذ. لا يوجد قائمة طعام في الغر...   
67125  الرسائل بين وائل و شوق كانت أجمل مافي الرواية....   
67126  استقبال سيء جدا وعدم الاستعداد للنزلاء . لا شي...   

                                              normalized  
0                             سيي جدا بعد الاصدار الجديد  
1                                      ابلكيشن زباله بجد  
2                                     سيي التطبيق لايعمل  
3      للاسف التطبيق للاسوا كان جدا رايع وس

**Tokenize Arabic words**

In [ ]:
import nltk
# Testing the function
text = 'إلى حياةٍ رائعة فيها مؤيد و مليئة بالكككككفاءة و شئون و رئيس هيئة'

text = normalize(text)

text = nltk.word_tokenize(text)
print(" ".join(text))

الي حياه رايعه فيها مويد و ملييه بالكفاءه و شيون و رييس هييه


In [ ]:
# Apply the function of normaliztion
df['tokens'] = df.apply(lambda row: nltk.word_tokenize(row['normalized']), axis=1)

df['tokens']

0                         [سيي, جدا, بعد, الاصدار, الجديد]
1                                    [ابلكيشن, زباله, بجد]
2                                   [سيي, التطبيق, لايعمل]
3        [للاسف, التطبيق, للاسوا, كان, جدا, رايع, وسهل,...
4                    [التحديث, بطيي, جدا, جدا, عند, الفتح]
                               ...                        
67122    [كتاب, جيد, وان, كان, مملا, بعض, الشيء, عند, م...
67123    [اول, تجربه, مع, الخيال, العلمي, ..., الكثير, ...
67124    [مرضي, ., الافطار, لذيذ, ., لا, يوجد, قايمه, ط...
67125    [الرسايل, بين, وايل, و, شوق, كانت, اجمل, مافي,...
67126    [استقبال, سيء, جدا, وعدم, الاستعداد, للنزلاء, ...
Name: tokens, Length: 67125, dtype: object

**Removing stop words**

In [107]:
arabic_stop_words = set([
    'في', 'من', 'إلى', 'عن', 'على', 'ما', 'لا', 'لم', 'لن', 'أن', 'إن', 'هذا', 'هذه',
    'هو', 'هي', 'كان', 'كانت', 'كما', 'لذلك', 'كل', 'أو', 'أي', 'بعض', 'ذلك', 'ثم', 'قد',
    'هناك', 'إذ', 'إذن', 'حيث', 'إلا', 'بين', 'مع', 'لكن', 'حتى', 'بعد', 'قبل', 'أكثر',
])

In [ ]:
def removing_stopwords(text):
    token = []
    for word in text:
        if word not in arabic_stop_words:
            token.append(word)
        
    return token

In [113]:
df['cleanTokens'] = df['tokens'].apply(removing_stopwords)
df['cleanTokens']

0                              [سيي, جدا, الاصدار, الجديد]
1                                    [ابلكيشن, زباله, بجد]
2                                   [سيي, التطبيق, لايعمل]
3        [للاسف, التطبيق, للاسوا, جدا, رايع, وسهل, وبسي...
4                    [التحديث, بطيي, جدا, جدا, عند, الفتح]
                               ...                        
67122    [كتاب, جيد, وان, مملا, الشيء, عند, منتصف, الكت...
67123    [اول, تجربه, الخيال, العلمي, ..., الكثير, المع...
67124    [مرضي, ., الافطار, لذيذ, ., يوجد, قايمه, طعام,...
67125    [الرسايل, وايل, و, شوق, اجمل, مافي, الروايه, ....
67126    [استقبال, سيء, جدا, وعدم, الاستعداد, للنزلاء, ...
Name: cleanTokens, Length: 67125, dtype: object

In [115]:
print(df[['tokens', 'cleanTokens']].head())

                                              tokens  \
0                   [سيي, جدا, بعد, الاصدار, الجديد]   
1                              [ابلكيشن, زباله, بجد]   
2                             [سيي, التطبيق, لايعمل]   
3  [للاسف, التطبيق, للاسوا, كان, جدا, رايع, وسهل,...   
4              [التحديث, بطيي, جدا, جدا, عند, الفتح]   

                                         cleanTokens  
0                        [سيي, جدا, الاصدار, الجديد]  
1                              [ابلكيشن, زباله, بجد]  
2                             [سيي, التطبيق, لايعمل]  
3  [للاسف, التطبيق, للاسوا, جدا, رايع, وسهل, وبسي...  
4              [التحديث, بطيي, جدا, جدا, عند, الفتح]  


In [ ]:
# Check if the column is clear from stop words or not
for tokens in df['cleanTokens']:
    if 'بعد' not in tokens:
        continue
    else:
        print('not clear')

In [ ]:
print('Before cleaning: ',df['tokens'].apply(len),'\n')
print('After cleaning: ',df['cleanTokens'].apply(len))

Before cleaning:  0         5
1         3
2         3
3        20
4         6
         ..
67122    11
67123    69
67124    17
67125    31
67126    39
Name: tokens, Length: 67125, dtype: int64 

After cleaning:  0         4
1         3
2         3
3        18
4         6
         ..
67122     9
67123    57
67124    14
67125    26
67126    37
Name: cleanTokens, Length: 67125, dtype: int64


In [132]:
# df['tokens'][3]

In [133]:
# df['cleanTokens'][3]

**Lemmatization**

In [143]:
from nltk.stem.snowball import SnowballStemmer


stemmer = SnowballStemmer("arabic")

def stemm(token):
    stemmed = []
    for word in token:
        stemmed_word = stemmer.stem(word)
        stemmed.append(stemmed_word)
        
    return stemmed

In [144]:
df['stemmed token'] = df['cleanTokens'].apply(stemm)
df['stemmed token']

0                                   [سي, جدا, اصدار, جديد]
1                                      [ابلكيش, زبال, بجد]
2                                       [سي, تطبيق, ايعمل]
3        [اسف, تطبيق, للاس, جدا, رايع, سهل, سيط, الان, ...
4                          [تحديث, بط, جدا, جدا, عند, فتح]
                               ...                        
67122      [كتاب, جيد, وان, ممل, شيء, عند, منتصف, كتاب, .]
67123    [اول, تجرب, خيال, علم, ..., كثير, معلوم, قالب,...
67124    [مرض, ., افطار, لذيذ, ., يوجد, قايم, طعام, غرف...
67125    [رسايل, وايل, و, شوق, اجمل, ماف, روايه, ..., ل...
67126    [استقبال, سيء, جدا, عدم, استعداد, نزلاء, ., شي...
Name: stemmed token, Length: 67125, dtype: object

**Rejoin tokens into clean string**

In [146]:
df['cleaned text'] = df['stemmed token'].apply(lambda tokens: ' '.join(tokens))
df['cleaned text']

0                                        سي جدا اصدار جديد
1                                          ابلكيش زبال بجد
2                                           سي تطبيق ايعمل
3        اسف تطبيق للاس جدا رايع سهل سيط الان معقد ولا ...
4                                 تحديث بط جدا جدا عند فتح
                               ...                        
67122                كتاب جيد وان ممل شيء عند منتصف كتاب .
67123    اول تجرب خيال علم ... كثير معلوم قالب ممتع . ق...
67124    مرض . افطار لذيذ . يوجد قايم طعام غرفهلم يتم ح...
67125    رسايل وايل و شوق اجمل ماف روايه ... لو انه مجر...
67126    استقبال سيء جدا عدم استعداد نزلاء . شيء . عدم ...
Name: cleaned text, Length: 67125, dtype: object

**TFIDF vectorization**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(df['cleaned text'])

**Split the data (train/val/test split)**

In [152]:
from sklearn.model_selection import train_test_split

y = df['rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train ,y_val = train_test_split(X_train, y_train, test_size= 0.2, random_state=42)

In [153]:
from sklearn.linear_model import LogisticRegression

logist = LogisticRegression()

logist.fit(X_train, y_train)

LogisticRegression()

In [155]:
val_pred = logist.predict(X_val)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_val,val_pred)

print('The accuracy on the validation for logistic model: {}'.format(accuracy))

The accuracy on the validation for logistic model: 0.873929236499069


In [160]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_val, val_pred))
print(confusion_matrix(y_val, val_pred))

              precision    recall  f1-score   support

          -1       0.88      0.89      0.89      3629
           0       0.92      0.85      0.88      3826
           1       0.82      0.89      0.85      3285

    accuracy                           0.87     10740
   macro avg       0.87      0.87      0.87     10740
weighted avg       0.88      0.87      0.87     10740

[[3233  132  264]
 [ 227 3244  355]
 [ 217  159 2909]]


In [ ]:
# To compare the performance between training and validation
train_pred = logist.predict(X_train)
accuracy_train = accuracy_score(y_train, train_pred)
print('The accuracy on the trainingset for logistic model: {}'.format(accuracy_train))

The accuracy on the trainingset for logistic model: 0.9245344506517691


In [162]:
# Testing
test_pred = logist.predict(X_test)
test_accuracy = accuracy_score(y_test, test_pred)

print('Testing accuracy: {}'.format(test_accuracy))

Testing accuracy: 0.8734450651769088


In [163]:
print(classification_report(y_test, test_pred))
print(confusion_matrix(y_test, test_pred))

              precision    recall  f1-score   support

          -1       0.88      0.89      0.88      4680
           0       0.93      0.85      0.88      4710
           1       0.82      0.88      0.85      4035

    accuracy                           0.87     13425
   macro avg       0.87      0.87      0.87     13425
weighted avg       0.88      0.87      0.87     13425

[[4172  137  371]
 [ 290 3993  427]
 [ 290  184 3561]]


**Cross-Validation (K-fold)**

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

logistModel = LogisticRegression()
kf = KFold(n_splits = 5, shuffle= True, random_state= 42)

accuracy = cross_val_score(logistModel, X,y, cv=kf, scoring= 'accuracy')
print("Cross-Validation Accuracy {}: ".format(accuracy))
print(f'Avg Accuracy: {np.mean(accuracy):.4f}')

Cross-Validation Accuracy [0.87463687 0.87024209 0.87642458 0.87798883 0.87247672]: 
Avg Accuracy: 0.8744
